# Time Series Analysis in Medicine and Biology
## Practical Course — University of Tübingen · PfeiferLab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamsaraE/time-series-medicine-biology/blob/main/teaching/06_white_red_noise_missingness.ipynb)

---

# Notebook 06 — White Noise, Red Noise, and Missing Data

**In this notebook you will:**
1. Generate and recognise **white noise** (no memory) via its flat ACF.
2. Build **red noise** as an **AR(1)** process and see how the correlation coefficient $r$ controls its smoothness.
3. Compare the **periodograms** of white vs. red noise (flat vs. low-frequency-heavy).
4. Create a series with **missing values** and compare common **imputation** methods.

**Data:** fully synthetic, so the notebook runs anywhere with no downloads.

**Why this matters:** "noise" is not one thing. Distinguishing memoryless white noise from
autocorrelated red noise tells you whether apparent patterns are real or just persistence — and
real biomedical series almost always arrive with gaps that must be handled before modelling.

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["axes.grid"] = True

## 2 · White noise

**White noise** is a sequence of independent draws with constant mean and variance and **no
memory**: the value at time $t$ tells you nothing about $t+1$. It is the null model for "no
structure" — what residuals should look like after a good model has removed all signal.

In [ ]:
# Set seed for reproducibility
np.random.seed(42)

# Length of time series
n = 200

# Generate white noise (mean=0, std=1)
white_noise = np.random.normal(loc=0, scale=1, size=n)

# Time axis
time = np.arange(n)

# Plot
plt.figure()
plt.plot(time, white_noise)
plt.title("White Noise Process")
plt.xlabel("Time")
plt.ylabel("Value")
plt.show()

### The ACF of white noise

The **autocorrelation function (ACF)** measures correlation between the series and lagged copies
of itself. For white noise, every lag beyond 0 should sit inside the confidence band — i.e. **no
significant autocorrelation**. This flat ACF is the visual signature of white noise.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(white_noise, lags=20)
plt.show()

## 3 · Red noise as an AR(1) process

**Red noise** has memory. We generate it as a first-order autoregressive — AR(1) — process:

$$x_t = r\,x_{t-1} + \sqrt{1-r^2}\;w_t,\qquad w_t \sim \mathcal{N}(0,\sigma^2)$$

The coefficient $r$ is the **lag-1 correlation**. The $\sqrt{1-r^2}$ factor keeps the variance
constant as $r$ changes, so we compare shapes fairly. Higher $r$ → smoother, more persistent.

In [ ]:
# Parameters
n = 200           # length of series
r = 0.7           # lag-1 correlation coefficient
sigma = 1         # standard deviation of white noise

# Generate white noise w_t
w = np.random.normal(loc=0, scale=sigma, size=n)

# Initialize red noise series
x = np.zeros(n)

# Generate AR(1) process
for t in range(1, n):
    x[t] = r * x[t-1] + np.sqrt(1 - r**2) * w[t]

# Time axis
time = np.arange(n)

# Plot
plt.figure()
plt.plot(time, x)
plt.title(f"Red Noise (AR(1)) Process, r = {r}")
plt.xlabel("Time")
plt.ylabel("Value")
plt.show()

### How $r$ changes the character of the series

Below we vary $r$ from 0 (pure white noise) to 0.95 (very persistent). As $r$ grows, the series
gets visibly **smoother** and wanders for longer before reverting — useful intuition for reading
real series that look "trendy" only because of strong short-term correlation.

In [ ]:
n = 300
time = np.arange(n)

r_values = [0.0, 0.4, 0.8, 0.95]

plt.figure(figsize=(10,8))

for i, r in enumerate(r_values):
    w = np.random.normal(size=n)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = r*x[t-1] + np.sqrt(1-r**2)*w[t]

    plt.subplot(len(r_values),1,i+1)
    plt.plot(time, x)
    plt.title(f"AR(1) with r = {r}")

plt.tight_layout()
plt.show()

## 4 · Spectral view: periodograms

The **periodogram** shows how variance is distributed across frequencies. White noise spreads its
power **evenly** across all frequencies (a flat spectrum — hence "white", like white light). Red
noise concentrates power at **low frequencies** (slow, large swings), with little high-frequency
content.

In [ ]:
from scipy.signal import periodogram


n = 100

# White noise
white = np.random.normal(size=n)

# Red noise (AR1)
r = 0.8
red = np.zeros(n)
w = np.random.normal(size=n)

for t in range(1, n):
    red[t] = r*red[t-1] + np.sqrt(1-r**2)*w[t]

# Compute periodograms
f_white, P_white = periodogram(white)
f_red, P_red = periodogram(red)

# Plot
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(f_white, P_white)
plt.title("White Noise Periodogram")

plt.subplot(1,2,2)
plt.plot(f_red, P_red)
plt.title("Red Noise (AR1) Periodogram")

plt.tight_layout()
plt.show()

**Red noise spectrum:** more power at low frequencies, decaying toward high frequencies — the
opposite of the flat white-noise spectrum.

## 5 · Missing data

Real biomedical series have gaps — a sensor drops out, a patient misses a visit, a lab value is
lost. Below we build a synthetic daily series (trend + weekly cycle + noise) and remove values in
two realistic ways: **scattered random gaps** and one **contiguous block** (e.g. a device offline
for two weeks).

In [ ]:
# 1) Create a synthetic time series with gaps (NaNs)


n = 240  # 240 days
idx = pd.date_range("2025-01-01", periods=n, freq="D")

# Signal: baseline + trend + weekly seasonality + noise
trend = 0.02 * np.arange(n)
season = 2.0 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = np.random.normal(scale=0.8, size=n)

ts = pd.Series(10 + trend + season + noise, index=idx, name="value")

# Introduce missingness: random missing points + one contiguous gap
ts_missing = ts.copy()
rng = np.random.default_rng(123)

random_missing_idx = rng.choice(n, size=20, replace=False)
ts_missing.iloc[random_missing_idx] = np.nan
ts_missing.loc["2025-04-10":"2025-04-25"] = np.nan  # contiguous gap



plt.figure(figsize=(12, 4))
plt.plot(ts_missing.index, ts_missing.values, linewidth=2)
plt.title("Synthetic time series with missing values (gaps)")
plt.xlabel("Time")
plt.ylabel("Value")
plt.grid(True, alpha=0.3)
plt.show()



### Visualising where the gaps are

A **missingness map** makes the pattern of gaps explicit — scattered points plus the solid block.
Seeing the structure of missingness matters: scattered gaps and long contiguous gaps call for
different handling.

In [ ]:
#Missingness map (yellow = missing)

plt.figure(figsize=(12, 1.3))
plt.imshow(ts_missing.isna().to_numpy()[None, :], aspect="auto")
plt.yticks([])
plt.title("Missingness map (yellow = missing)")
plt.xlabel("Time index (days)")
plt.show()


## 6 · Imputation methods compared

We compare four standard approaches:

- **Forward fill (LOCF)** — carry the last observed value forward. Simple, but creates flat steps.
- **Time interpolation (linear)** — draw a straight line across the gap, respecting the dates.
- **Spline (order 3)** — smooth cubic fit; can **overshoot** across long gaps.

Watch how they differ most across the **long contiguous gap** — that is where the choice of method
matters, and where all of them are least trustworthy.

In [ ]:

# Missing-data handling methods

ts_ffill = ts_missing.ffill()
ts_bfill = ts_missing.bfill()
ts_linear = ts_missing.interpolate(method="time")  # time-aware linear interpolation
ts_spline = ts_missing.interpolate(method="spline", order=3)  # may overshoot; for comparison

plt.figure(figsize=(12, 4))
plt.plot(ts_missing.index, ts_missing.values, label="Original (with NaNs)", linewidth=2)
plt.plot(ts_ffill.index, ts_ffill.values, label="Forward fill (LOCF)", alpha=0.9)
plt.plot(ts_linear.index, ts_linear.values, label="Time interpolation (linear)", alpha=0.9)
plt.plot(ts_spline.index, ts_spline.values, label="Spline (order=3)", alpha=0.8)
plt.title("Missing-data handling: overlay comparison")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Key takeaways

- **White noise** = no memory → flat ACF, flat spectrum. It is the target shape for model residuals.
- **Red noise (AR(1))** has memory controlled by $r$; higher $r$ means smoother, more persistent series and a low-frequency-heavy spectrum.
- **No single imputation is "correct"**: forward-fill creates steps, linear interpolation is neutral, splines can overshoot. The longer the gap, the less any method should be trusted.

**Try it yourself:**
1. Set $r = -0.7$ in the AR(1) process — what does *anti-persistent* noise look like?
2. Plot the ACF of the red-noise series; how many lags stay significant?
3. Add a longer contiguous gap and see which imputation degrades most.